# Lab 4: Curadoria de Dataset

## Lab 4: Curadoria de Dataset — do dado bruto ao formato de treino

Usamos o [Databricks Dolly 15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k)
(CC BY-SA 3.0) — 15 mil instruções reais, escritas por humanos, não
geradas por LLM. Pra demonstrar limpeza de dados de forma honesta,
injetamos "sujeira" artificial (duplicatas, vazios) numa cópia dos dados
reais — o texto em si é real, a bagunça é simulada pra fins didáticos.

In [1]:
!pip install -q datasets transformers

from datasets import load_dataset
import random

raw = load_dataset("databricks/databricks-dolly-15k", split="train[:200]")
print(f"✓ {len(raw)} exemplos reais carregados (Databricks Dolly 15k)")
print(raw[0])

✓ 200 exemplos reais carregados (Databricks Dolly 15k)
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


### 1. Simulando um dataset "sujo" (pra praticar limpeza)

In [2]:
random.seed(42)
dirty = list(raw)
# duplicatas
dirty += random.sample(dirty, 15)
# exemplos vazios/quebrados
dirty += [{"instruction": "", "context": "", "response": "", "category": "broken"} for _ in range(5)]
dirty += [{"instruction": "oi", "context": "", "response": "", "category": "broken"} for _ in range(3)]
random.shuffle(dirty)

print(f"Dataset 'sujo': {len(dirty)} exemplos ({len(dirty) - len(raw)} a mais que o original — duplicatas + quebrados)")

Dataset 'sujo': 223 exemplos (23 a mais que o original — duplicatas + quebrados)


**Resultado esperado:** `Dataset 'sujo': 223 exemplos (23 a mais que o
original)` — 200 reais + 15 duplicatas + 8 quebrados.

### 2. Limpeza: remover vazios, curtos demais, e deduplicar

In [3]:
def clean_dataset(examples):
    seen = set()
    cleaned = []
    for ex in examples:
        instruction = ex["instruction"].strip()
        response = ex["response"].strip()

        if not instruction or not response:
            continue
        if len(instruction) < 5 or len(response) < 5:
            continue

        key = (instruction, response)
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(ex)
    return cleaned

cleaned = clean_dataset(dirty)
print(f"Antes da limpeza: {len(dirty)} exemplos")
print(f"Depois da limpeza: {len(cleaned)} exemplos")
print(f"Removidos: {len(dirty) - len(cleaned)} (vazios/quebrados + duplicatas)")

Antes da limpeza: 223 exemplos
Depois da limpeza: 199 exemplos
Removidos: 24 (vazios/quebrados + duplicatas)


**Resultado esperado:** volta pra próximo de 200 (os 23 exemplos sujos
injetados são removidos — algumas duplicatas podem já existir no dataset
original, então o número exato pode variar ligeiramente).

### 3. Formatando com um chat template real

**Por que importa:** cada família de modelo espera um formato específico
de delimitação entre turnos — usar o template certo é obrigatório pra
fine-tuning funcionar bem.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

def format_example(ex):
    messages = [{"role": "user", "content": ex["instruction"]}]
    if ex["context"]:
        messages[0]["content"] += f"\n\nContexto: {ex['context']}"
    messages.append({"role": "assistant", "content": ex["response"]})
    return tokenizer.apply_chat_template(messages, tokenize=False)

formatted = [format_example(ex) for ex in cleaned[:3]]
for f in formatted:
    print(f)
    print("=" * 60)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is Sunshine Recession?<|im_end|>
<|im_start|>assistant
It is known as the deepest period in which sunspots are not virtually visible. Deepest period is related to sun cycle's process called solar minimum<|im_end|>

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
How do you operate a car with a manual transmission?<|im_end|>
<|im_start|>assistant
Through a combination of a shifter and three pedals: gas, brake, and clutch. Press down the clutch pedal with one foot, and the brake with your other foot first and then turn on the engine of the car. It is recommended to shift to neutral immediately after turning on your car and prior to departing. To shift between one of the engaged gears, typically marked 1, 2, 3, 4, 5 N (for neutral) and R (for reverse), press down on the clutch all the way to the floor and

**Resultado esperado:** cada exemplo aparece envolto nos tokens especiais
do chat template do SmolLM2 (algo como `<|im_start|>user ... <|im_end|>
<|im_start|>assistant ...`) — o texto cru que o modelo realmente vê durante
o treino.

### 4. Split treino/validação e tokenização final

In [5]:
random.shuffle(cleaned)
split_idx = int(len(cleaned) * 0.85)
train_set, val_set = cleaned[:split_idx], cleaned[split_idx:]

print(f"Treino: {len(train_set)} exemplos")
print(f"Validação: {len(val_set)} exemplos")

sample_tokens = tokenizer(format_example(train_set[0]), return_tensors="pt")
print(f"\nTokens do primeiro exemplo de treino: {sample_tokens['input_ids'].shape[1]}")
print(f"Attention mask (deveria ser tudo 1, sem padding neste exemplo isolado): {sample_tokens['attention_mask'].sum().item()}")

Treino: 169 exemplos
Validação: 30 exemplos

Tokens do primeiro exemplo de treino: 331
Attention mask (deveria ser tudo 1, sem padding neste exemplo isolado): 331


**Resultado esperado:** split ~85/15 real, e a tokenização confirma que o
attention mask soma exatamente o número de tokens (sem padding quando
processamos um exemplo isolado — padding só entra quando formamos um
batch com sequências de tamanhos diferentes, Semana 4.5).

**Próximos passos:** `train_set` e `val_set` daqui são exatamente o que o
Lab 5 (SFT) usa pra fine-tuning de verdade.